### Resolve Imports

In [2]:
from datasets import load_dataset
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

### Import dataset

In [4]:
dataset = load_dataset("Annanay/aml_song_lyrics_balanced")
data = dataset['train']


df = pd.DataFrame(data)

df = df[['lyrics', 'mood', 'mood_cats']]

### Preprocess text

In [5]:
def preprocess_text(text):
    text = re.sub(r'\[.*>\]', '', text)  # Remove text within brackets
    text = re.sub(r'^a-zA-Z\s', '', text)  # Remove non-alphabetic characters
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text = text.lower().strip()  # Remove leading and trailing spaces
    return text

### Save cleaned up lyrics

In [6]:
df['clean_lyrics'] = df['lyrics'].apply(preprocess_text)

### Train-test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_lyrics'],  # the input features (lyrics)
    df['mood'],          # the target labels (happy, sad, etc.)
    test_size=0.2,       # 20% of data will be used for testing
    random_state=42       # ensures reproducibility (same split every time)
)

`train_test_split` **randomly shuffles** your data rows, then:

* Takes **80%** of lyrics + moods → goes to `X_train` and `y_train`
* Takes **20%** → goes to `X_test` and `y_test`

So in the end:

| Variable  | Meaning             | Example size (if 1000 songs) |
| --------- | ------------------- | ---------------------------- |
| `X_train` | training lyrics     | 800                          |
| `y_train` | moods for those 800 | 800                          |
| `X_test`  | testing lyrics      | 200                          |
| `y_test`  | moods for those 200 | 200                          |

### TF-IDF Vectorization

In [8]:
vectorizer = TfidfVectorizer(
    max_features=5000,  # limit vocabulary size
    stop_words='english',  # remove common words
    ngram_range=(1,2)  # include unigrams + bigrams
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Checking Output:

In [9]:
print("✅ TF-IDF ready!")
print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

✅ TF-IDF ready!
Train shape: (9756, 5000)
Test shape: (2440, 5000)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize the model
model = LogisticRegression(max_iter=1000, solver='lbfgs')

# Train the model on the TF-IDF vectors
model.fit(X_train_tfidf, y_train)

# Predict moods on the test data
y_pred = model.predict(X_test_tfidf)

# Evaluate accuracy
print("✅ Model trained successfully!")
print("Accuracy:", accuracy_score(y_test, y_pred))

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Optional: see confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

✅ Model trained successfully!
Accuracy: 0.603688524590164

Classification Report:
              precision    recall  f1-score   support

       anger       0.76      0.90      0.82       598
        calm       0.60      0.66      0.63       598
       happy       0.53      0.49      0.51       623
         sad       0.48      0.38      0.43       621

    accuracy                           0.60      2440
   macro avg       0.59      0.61      0.60      2440
weighted avg       0.59      0.60      0.59      2440


Confusion Matrix:
[[536  17  30  15]
 [ 34 395  81  88]
 [ 48 114 304 157]
 [ 89 132 162 238]]
